# PMC Embedding Pipeline — LangChain + LangGraph

用 `langchain-huggingface` 管理 BGE 嵌入模型，`langchain-chroma` 管理向量库，`langgraph` 搭建有状态的批次处理管道（含断点续传）。

## 0. 安装依赖

In [ ]:
# 只需运行一次
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'langchain-huggingface',
    'langchain-chroma',
    'langgraph',
], check=True)
print('安装完成')

## 1. 配置

In [ ]:
import os, sys
os.environ.setdefault('OMP_NUM_THREADS',   str(os.cpu_count() or 4))
os.environ.setdefault('HF_ENDPOINT',       'https://hf-mirror.com')
os.environ['PYTORCH_CUDA_ALLOC_CONF']    = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']     = 'false'
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

from pathlib import Path
import subprocess

# ── 探测批次文件位置 ─────────────────────────────────────────────
print('=== 探测批次文件位置 ===')
search_roots = [Path('/root/autodl-tmp'), Path('/root'), Path.home()]
found_dirs = []
for root in search_roots:
    if not root.exists():
        continue
    for m in root.rglob('batch_00000000.parquet'):
        resolved = m.parent.resolve()   # 确保绝对路径
        if resolved not in found_dirs:
            found_dirs.append(resolved)
            print(f'  找到: {resolved}')
if not found_dirs:
    result = subprocess.run(['find', '/root/autodl-tmp', '-name', '*.parquet',
                             '-maxdepth', '5'], capture_output=True, text=True)
    print(result.stdout[:1000] or '  未找到 parquet 文件，请手动填写 BATCH_DIR')
print()

# ── 修改这里 ─────────────────────────────────────────────────────
BATCH_DIR    = found_dirs[0] if found_dirs else Path('/root/autodl-tmp/batches_full')
DB_DIR       = BATCH_DIR.parent / 'pipeline_output' / 'chroma_db'  # 与已有 ChromaDB 保持一致
COLLECTION   = 'pmc_full'
MODEL        = 'BAAI/bge-base-en-v1.5'
EMBED_BATCH  = 2048   # OOM 时退至 1024/512
CHROMA_BATCH = 5000
RESUME       = True
# ─────────────────────────────────────────────────────────────────

batch_files = sorted(BATCH_DIR.glob('batch_*.parquet'))
print(f'BATCH_DIR  : {BATCH_DIR}')
print(f'DB_DIR     : {DB_DIR}')
print(f'批次文件数 : {len(batch_files):,}')
if batch_files:
    print(f'第一个     : {batch_files[0].name}')
    print(f'最后一个   : {batch_files[-1].name}')
else:
    print('⚠️  批次文件数为 0，请检查 BATCH_DIR')

## 2. 初始化 LangChain Embedder + VectorStore

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# langchain-huggingface 把 model_kwargs 直接解包给 SentenceTransformer.__init__()
# torch_dtype 不是顶层参数，改为加载后手动转 fp16
embedder = HuggingFaceEmbeddings(
    model_name   = MODEL,
    model_kwargs = {'device': device},
    encode_kwargs= {'normalize_embeddings': True, 'batch_size': EMBED_BATCH},
)

# 转 fp16：节省约 50% 显存，推理速度更快
if device == 'cuda':
    embedder._client.half()
    print('模型已转换为 fp16')

print(f'嵌入模型加载完成  维度={len(embedder.embed_query("test"))}')
if device == 'cuda':
    torch.cuda.empty_cache()

DB_DIR.mkdir(parents=True, exist_ok=True)
vectorstore = Chroma(
    collection_name    = COLLECTION,
    embedding_function = embedder,
    persist_directory  = str(DB_DIR),
    collection_metadata= {'hnsw:space': 'cosine'},
)
print(f'ChromaDB 集合已有向量: {vectorstore._collection.count():,}')

## 3. 定义 LangGraph Pipeline

图结构：
```
START → load_batch → embed_store → checkpoint → (更多批次?) → load_batch
                                                            ↓
                                                           END
```

In [ ]:
import json, time, threading, queue
import pandas as pd
import numpy as np
from typing import TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

CKPT_PATH = BATCH_DIR / f'lc_embed_checkpoint_{COLLECTION}.json'

def load_checkpoint() -> set:
    if CKPT_PATH.exists():
        return set(json.loads(CKPT_PATH.read_text()))
    return set()

def save_checkpoint(done: set):
    CKPT_PATH.write_text(json.dumps(sorted(done)))

# ── Pipeline 状态 ────────────────────────────────────────────────
class EmbedState(TypedDict):
    batch_idx  : int
    done_files : set
    total_new  : int
    t_start    : float
    current_df : object

# ── 元数据向量化构建 ─────────────────────────────────────────────
STR_FIELDS = ['doc_id','chunk_type','section_title','section_path','imrad_type',
               'split_strategy','pmc_id','pmid','doi','journal','article_type',
               'abstract_source','source_url','source_title']
INT_FIELDS = ['pub_year','token_count','chunk_index','total_chunks']

def batch_make_metadata(df: pd.DataFrame) -> list[dict]:
    tmp = {}
    for f in STR_FIELDS:
        tmp[f] = df[f].fillna('').astype(str).tolist() if f in df.columns else ['']*len(df)
    for f in INT_FIELDS:
        tmp[f] = pd.to_numeric(df[f], errors='coerce').fillna(0).astype(int).tolist() \
                 if f in df.columns else [0]*len(df)
    all_f = STR_FIELDS + INT_FIELDS
    return [{f: tmp[f][i] for f in all_f} for i in range(len(df))]

# ── 节点 1：读取批次（IO 线程预读）──────────────────────────────
_prefetch_q: queue.Queue = queue.Queue(maxsize=2)

def _start_prefetch(files: list):
    def _reader():
        for f in files:
            try:
                _prefetch_q.put((f, pd.read_parquet(f)))
            except Exception as e:
                _prefetch_q.put((f, None))
        _prefetch_q.put(None)
    threading.Thread(target=_reader, daemon=True).start()

def node_load(state: EmbedState) -> EmbedState:
    item = _prefetch_q.get()
    if item is None:
        return {**state, 'current_df': None, 'batch_idx': len(batch_files)}
    bf, df = item
    return {**state, 'current_df': df}

# ── 节点 2：先 embed 全批，再集中写入 ChromaDB ───────────────────
def node_embed_store(state: EmbedState) -> EmbedState:
    df = state['current_df']
    if df is None or df.empty:
        return {**state, 'current_df': None}

    texts     = df['text'].tolist()
    ids       = df['chunk_id'].tolist()
    metadatas = batch_make_metadata(df)

    # ① GPU 连续 encode 整批
    all_embeddings = embedder.embed_documents(texts)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ② ChromaDB 批量写入（跳过 LangChain 二次 embed）
    for start in range(0, len(df), CHROMA_BATCH):
        end = min(start + CHROMA_BATCH, len(df))
        for attempt in range(5):
            try:
                vectorstore._collection.add(
                    ids        = ids[start:end],
                    embeddings = all_embeddings[start:end],
                    documents  = texts[start:end],
                    metadatas  = metadatas[start:end],
                )
                break
            except Exception as e:
                if attempt == 4: raise
                wait = 5 * (2 ** attempt)
                print(f'  ChromaDB 写入失败 ({attempt+1}/5)，{wait}s 后重试: {e}')
                time.sleep(wait)

    return {**state, 'current_df': None, 'total_new': state['total_new'] + len(df)}

# ── 节点 3：断点保存 + 每批打印进度 ─────────────────────────────
def node_checkpoint(state: EmbedState) -> EmbedState:
    idx  = state['batch_idx']
    bf   = batch_files[idx]
    done = state['done_files'] | {str(bf)}
    save_checkpoint(done)

    elapsed  = time.time() - state['t_start']
    speed    = state['total_new'] / elapsed if elapsed > 0 else 0
    eta_h    = (len(batch_files) - idx - 1) * (state['total_new'] / max(idx + 1, 1)) \
               / speed / 3600 if speed > 0 else 0

    ts = time.strftime('%H:%M:%S')
    print(f'  {ts}  [{idx+1:>4}/{len(batch_files)}]  {bf.name}  '
          f'累计={state["total_new"]:,}  '
          f'{speed:.0f} c/s  '
          f'ETA={eta_h:.1f}h')

    return {**state, 'batch_idx': idx + 1, 'done_files': done}

# ── 条件边 ───────────────────────────────────────────────────────
def should_continue(state: EmbedState) -> str:
    idx = state['batch_idx']
    while idx < len(batch_files) and str(batch_files[idx]) in state['done_files']:
        idx += 1
    return END if idx >= len(batch_files) else 'load'

# ── 构建图 ───────────────────────────────────────────────────────
graph = StateGraph(EmbedState)
graph.add_node('load',        node_load)
graph.add_node('embed_store', node_embed_store)
graph.add_node('checkpoint',  node_checkpoint)
graph.set_entry_point('load')
graph.add_edge('load',        'embed_store')
graph.add_edge('embed_store', 'checkpoint')
graph.add_conditional_edges('checkpoint', should_continue,
                             {'load': 'load', END: END})
pipeline = graph.compile()
print('LangGraph pipeline 构建完成')

## 4. 运行 Pipeline

In [ ]:
done_files = load_checkpoint() if RESUME else set()

# 找第一个未完成的批次
start_idx = 0
while start_idx < len(batch_files) and str(batch_files[start_idx]) in done_files:
    start_idx += 1

print(f'已完成批次: {len(done_files):,} / {len(batch_files):,}')
print(f'从批次索引 {start_idx} 开始')
print(f'开始时间  : {time.strftime("%Y-%m-%d %H:%M:%S")}')
print('─' * 60)

if start_idx >= len(batch_files):
    print('所有批次已完成！')
else:
    pending = batch_files[start_idx:]

    # 启动 IO 预读线程（在 GPU encode 时提前读取下一批 parquet）
    import queue as _q
    _prefetch_q.queue.clear()   # 清空旧队列
    _start_prefetch(pending)

    t0 = time.time()
    final_state = pipeline.invoke(
        {
            'batch_idx' : start_idx,
            'done_files': done_files,
            'total_new' : 0,
            't_start'   : t0,
            'current_df': None,
        },
        config={'recursion_limit': len(batch_files) + 10},
    )

    elapsed = time.time() - t0
    print('─' * 60)
    print(f'完成时间  : {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'总耗时    : {elapsed/3600:.2f} 小时')
    print(f'本次新增  : {final_state["total_new"]:,}')
    print(f'集合总向量: {vectorstore._collection.count():,}')

## 5. 两阶段模式（推荐：先缓存 Embedding，再写入 ChromaDB）

与 Cell 4 的 LangGraph pipeline 二选一运行。

| 阶段 | 做什么 | 需要 GPU？ |
|------|--------|-----------|
| **Phase 1**（下方第一个 Cell） | parquet → encode → 保存 `.npy`（float16） | ✅ 需要 |
| **Phase 2**（下方第二个 Cell） | `.npy` → ChromaDB → 删除 `.npy` | ❌ 不需要 |

**优势**：GPU encode 不被 ChromaDB HNSW 写入打断，跑完 Phase 1 后可换 CPU 节点执行 Phase 2。  
**磁盘占用**：每批约 71 MB（float16），200 批 ≈ 14 GB。

In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase 1：纯 GPU Embedding，结果保存为 .npy 缓存
# 前置条件：已运行 Cell 1（配置）、Cell 2（模型加载）、Cell 3（LangGraph 定义）
# ══════════════════════════════════════════════════════════════

CACHE_DIR = BATCH_DIR.parent / 'emb_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── checkpoint 丢失/路径不对时：改为 True，从 ChromaDB 重建 ────
REBUILD_FROM_CHROMA = False
# ─────────────────────────────────────────────────────────────

# ── checkpoint 加载 ──────────────────────────────────────────
chroma_done = load_checkpoint()
print(f'checkpoint 路径  : {CKPT_PATH}')
print(f'checkpoint 存在  : {CKPT_PATH.exists()}')
print(f'已完成批次数     : {len(chroma_done):,}')
if chroma_done:
    sample = sorted(chroma_done)[:2]
    print(f'  示例路径       : {sample[0]}')
if batch_files:
    print(f'  batch_files[0] : {str(batch_files[0])}')
print()

if REBUILD_FROM_CHROMA or (not CKPT_PATH.exists() and vectorstore._collection.count() > 0):
    print(f'从 ChromaDB 重建 checkpoint（当前向量数: {vectorstore._collection.count():,}）')
    print(f'共 {len(batch_files):,} 个批次，逐批探测...')
    rebuilt = set()
    for i, bf in enumerate(batch_files):
        try:
            first_id = pd.read_parquet(bf, columns=['chunk_id'])['chunk_id'].iloc[0]
            result   = vectorstore._collection.get(ids=[str(first_id)], include=[])
            if result['ids']:
                rebuilt.add(str(bf))
        except Exception as e:
            print(f'  ⚠️  {bf.name} 探测失败: {e}')
        if (i + 1) % 200 == 0 or (i + 1) == len(batch_files):
            print(f'  [{i+1}/{len(batch_files)}]  已确认完成: {len(rebuilt):,}')
    save_checkpoint(rebuilt)
    chroma_done = rebuilt
    print(f'重建完成：{len(chroma_done):,} / {len(batch_files):,} 批次已在 ChromaDB 中')
    print('─' * 60)

# ── 待处理列表 ───────────────────────────────────────────────
pending_p1 = [
    f for f in batch_files
    if str(f) not in chroma_done
    and not (CACHE_DIR / f'{f.stem}_emb.npy').exists()
]

print(f'EMBED_BATCH     : {EMBED_BATCH}')
print(f'已写入 ChromaDB : {len(chroma_done):,}')
print(f'本次待 Embed    : {len(pending_p1):,}')
print(f'预计磁盘占用    : {len(pending_p1) * 46435 * 768 * 2 / 1e9:.1f} GB')
print('─' * 60)

if not pending_p1:
    print('无需 Embed，请直接运行 Phase 2。')
else:
    import queue as _p1q, threading as _thr
    _p1_queue: _p1q.Queue = _p1q.Queue(maxsize=4)

    def _p1_reader(files, q):
        for f in files:
            try:
                q.put((f, pd.read_parquet(f)))
            except Exception as e:
                q.put((f, None))
        q.put(None)

    _thr.Thread(target=_p1_reader, args=(pending_p1, _p1_queue), daemon=True).start()

    t0, total_done = time.time(), 0

    for i in range(len(pending_p1)):
        item = _p1_queue.get()
        if item is None:
            break
        bf, df = item
        if df is None:
            print(f'  ⚠️  读取失败，跳过: {bf.name}')
            continue

        texts = df['text'].tolist()

        with torch.inference_mode():
            embs = embedder._client.encode(
                texts,
                batch_size           = EMBED_BATCH,
                normalize_embeddings = True,
                show_progress_bar    = False,
                convert_to_numpy     = True,
            )
        embs = embs.astype(np.float16)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        np.save(CACHE_DIR / f'{bf.stem}_emb.npy', embs)
        total_done += len(texts)

        elapsed = time.time() - t0
        speed   = total_done / elapsed
        eta_h   = (len(pending_p1) - i - 1) * (total_done / max(i+1,1)) / speed / 3600
        ts      = time.strftime('%H:%M:%S')
        print(f'  {ts}  [{i+1:>4}/{len(pending_p1)}]  {speed:.0f} c/s  ETA={eta_h:.1f}h')

    print('─' * 60)
    print(f'Phase 1 完成！共 embed {total_done:,} chunks')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase 2：从 .npy 缓存写入 ChromaDB，写入后删除缓存文件
# 前置条件：已运行 Cell 3（LangGraph 定义，需要 batch_make_metadata / save_checkpoint）
# 不需要 GPU，可在 CPU 节点上运行（只需能访问 CACHE_DIR 和 DB_DIR）
# ══════════════════════════════════════════════════════════════

chroma_done = load_checkpoint()
cache_files = sorted(CACHE_DIR.glob('*_emb.npy'))

pending_p2 = []
for npy in cache_files:
    stem         = npy.stem.replace('_emb', '')
    parquet_file = BATCH_DIR / f'{stem}.parquet'
    if str(parquet_file) in chroma_done:
        npy.unlink()
        print(f'  清理残留缓存: {npy.name}')
        continue
    if not parquet_file.exists():
        print(f'  ⚠️  找不到原始 parquet: {parquet_file.name}，跳过')
        continue
    pending_p2.append((npy, parquet_file))

print(f'待写入 ChromaDB : {len(pending_p2):,}')
print(f'已完成批次      : {len(chroma_done):,}')
print(f'ChromaDB 当前量 : {vectorstore._collection.count():,}')
print('─' * 60)

if not pending_p2:
    print('无缓存文件待写入。请先运行 Phase 1。')
else:
    t0, total_written = time.time(), 0

    for i, (npy_file, parquet_file) in enumerate(pending_p2):
        embs = np.load(npy_file).astype(np.float32)

        # 与 Phase 1 保持相同顺序（不排序）
        df        = pd.read_parquet(parquet_file)
        texts     = df['text'].tolist()
        ids       = df['chunk_id'].tolist()
        metadatas = batch_make_metadata(df)

        if len(embs) != len(df):
            print(f'  ⚠️  {npy_file.name}: 向量数 {len(embs)} ≠ chunk 数 {len(df)}，跳过')
            continue

        success = True
        for start in range(0, len(df), CHROMA_BATCH):
            end = min(start + CHROMA_BATCH, len(df))
            for attempt in range(5):
                try:
                    vectorstore._collection.add(
                        ids        = ids[start:end],
                        embeddings = embs[start:end].tolist(),
                        documents  = texts[start:end],
                        metadatas  = metadatas[start:end],
                    )
                    break
                except Exception as e:
                    if attempt == 4:
                        print(f'  ChromaDB 写入失败（已重试 5 次）: {e}')
                        success = False
                        break
                    wait = 5 * (2 ** attempt)
                    print(f'  写入失败 ({attempt+1}/5)，{wait}s 后重试: {e}')
                    time.sleep(wait)
            if not success:
                break

        if success:
            chroma_done.add(str(parquet_file))
            save_checkpoint(chroma_done)
            npy_file.unlink()

            total_written += len(df)
            elapsed = time.time() - t0
            speed   = total_written / elapsed if elapsed > 0 else 0
            eta_h   = (len(pending_p2) - i - 1) * (total_written / max(i+1,1)) / speed / 3600 if speed > 0 else 0
            ts      = time.strftime('%H:%M:%S')
            print(f'  {ts}  [{i+1:>4}/{len(pending_p2)}]  {parquet_file.name}  '
                  f'累计={total_written:,}  ETA={eta_h:.1f}h')

    print(f'\nPhase 2 完成！写入 {total_written:,} chunks')
    print(f'ChromaDB 集合总向量: {vectorstore._collection.count():,}')
    print(f'剩余缓存文件: {len(list(CACHE_DIR.glob("*_emb.npy")))}（应为 0）')

## 5. 查询验证

In [ ]:
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def search(query: str, k: int = 5, filter: dict = None):
    """语义检索，自动加 BGE 查询前缀。"""
    prefixed = BGE_QUERY_PREFIX + query
    results  = vectorstore.similarity_search_with_score(
        query  = prefixed,
        k      = k,
        filter = filter,
    )
    print(f'Query: {query}')
    for i, (doc, score) in enumerate(results):
        sim  = 1 - score   # chromadb 返回余弦距离，转为相似度
        meta = doc.metadata
        print(f'  [{i+1}] sim={sim:.4f}  '
              f'type={meta.get("chunk_type","")}  '
              f'imrad={meta.get("imrad_type","")}  '
              f'year={meta.get("pub_year","")}  '
              f'journal={meta.get("journal","")[:30]}')
        print(f'       {doc.page_content[:120]}')
    print()

search('CRISPR gene editing therapy')
search('COVID-19 vaccine efficacy clinical trial')
search('machine learning drug discovery')

In [ ]:
# 元数据过滤查询
search(
    query  = 'RNA sequencing single cell analysis protocol',
    k      = 5,
    filter = {'imrad_type': 'methods', 'pub_year': {'$gte': 2020}},
)